# Robot zbierający puszki. Od tabeli do symulacji

Otwórz ten plik podczas wspólnego kodowania, po zadaniach na kartce.
Uzupełnij jedną funkcję i porównaj jej wyniki ze swoim rachunkiem.
Uruchom komórki od góry. Po zmianie funkcji wykonaj ponownie komórkę
z funkcją i komórkę z testami.

## 1. Dane modelu ze slajdu

**H** oznacza wysoką energię, a **L** niską. Przy szukaniu pierwsza gałąź
zachowuje poziom energii. Druga prowadzi z H do L, a z L do H po ratunku.
Ratunek kosztuje 3 jednostki nagrody. Ładowanie jest możliwe w L i prowadzi
do H z nagrodą 0. Czekanie zachowuje poziom energii.

W przykładzie z zajęć α = 0,50 to szansa pozostania w H po szukaniu,
a β = 0,50 to szansa pozostania w L po szukaniu. Średnia nagroda za szukanie
bez ratunku wynosi 2, a za czekanie 0,50. Każda gałąź w kodzie to
`(prawdopodobieństwo, następny_stan, nagroda)`.

In [ ]:
H, L = 0, 1
STATE_NAMES = {H: "H", L: "L"}
SEARCH, WAIT, RECHARGE = "szukaj", "czekaj", "ładuj"

def build_model(alpha=0.50, beta=0.50, r_search=2.0, r_wait=0.50):
    if not (0.0 <= alpha <= 1.0 and 0.0 <= beta <= 1.0):
        raise ValueError("Prawdopodobieństwa muszą należeć do [0, 1].")
    return {
        (H, SEARCH): [(alpha, H, r_search), (1 - alpha, L, r_search)],
        (H, WAIT): [(1.0, H, r_wait), (0.0, L, r_wait)],
        (L, SEARCH): [(beta, L, r_search), (1 - beta, H, -3.0)],
        (L, WAIT): [(1.0, L, r_wait), (0.0, H, r_wait)],
        (L, RECHARGE): [(1.0, H, 0.0), (0.0, L, 0.0)],
    }

model = build_model()

## 2. Od liczby u do wyniku akcji

Funkcja `step_robot(branches, u)` dostaje uporządkowane gałęzie **jednej** akcji
w **jednym** stanie i zwraca parę `(następny_stan, nagroda)`.
`u` to liczba z przedziału **[0, 1)**. W symulacji losujemy ją jednostajnie,
a w teście podajemy konkretne `u`, żeby odtworzyć ten sam wynik.

Każda gałąź zajmuje przedział długości równej jej prawdopodobieństwu.
Sumuj prawdopodobieństwa po kolei, aż suma będzie **większa** od `u`.
Jeśli `u` trafia dokładnie na granicę, przejdź dalej. Gałąź o zerowym
prawdopodobieństwie ma pusty przedział.

Przed kodowaniem zaznacz na odcinku [0, 1) wartości 0,49 i 0,50.
Odczytaj, co każda z nich oznacza dla szukania w H.

### Wskazówka. Przełóż tabelę na pseudokod

Wyobraź sobie dwie gałęzie o szansach 0,25 i 0,75. Pierwsza zajmuje
[0; 0,25), a druga [0,25; 1). Dla `u = 0.25` wybierasz drugą gałąź.
Przewidź, co się stanie, jeśli pierwsza gałąź będzie miała szansę 0
i podstawisz `u = 0`.

1. Zacznij od sumy prawdopodobieństw równej 0.
2. Przejdź po trójkach z wiersza tabeli w podanej kolejności.
   Po odczytaniu trójki dodaj jej prawdopodobieństwo do sumy.
3. Gdy suma po raz pierwszy jest ściśle większa od `u`, zwróć
   parę ze stanem i nagrodą z tej trójki.

Zastąp `raise NotImplementedError` własnym kodem. Dane wejściowe
zawierają już `u` i gałęzie wybranej akcji.

In [ ]:
def step_robot(branches, u):
    # TODO 1. Ustaw początkową sumę prawdopodobieństw.
    # TODO 2. Przejdź po gałęziach. Po każdej zaktualizuj sumę.
    # TODO 3. Przy pierwszym ścisłym przekroczeniu u zwróć parę.
    raise NotImplementedError("Uzupełnij losowanie wyniku kroku.")

## 3. Sprawdź funkcję na pojedynczych ruchach

Testy obejmują zwykły wynik, granicę przedziału, ratunek, ładowanie
oraz gałąź o zerowym prawdopodobieństwie. Gdy wszystkie przejdą,
odtwórz cały przebieg.

In [ ]:
assert step_robot(model[H, SEARCH], 0.49) == (H, 2.0), "Szukaj w H, u=0,49"
assert step_robot(model[H, SEARCH], 0.50) == (L, 2.0), "Szukaj w H, u=0,50"
assert step_robot(model[L, SEARCH], 0.90) == (H, -3.0), "Ratunek po szukaniu w L"
assert step_robot(model[L, RECHARGE], 0.999) == (H, 0.0), "Ładowanie w L"
assert step_robot(build_model(alpha=0.0)[H, SEARCH], 0.0) == (L, 2.0), "Gałąź o p=0"

print("Pięć testów przechodzi.")

## 4. Odtwórz trzy ruchy z kartki

Zacznij w H. Podajemy te same akcje i liczby `u` co w zadaniu z prezentacji.
Po każdym ruchu następny stan staje się początkiem kolejnego kroku.
W czekaniu wynik jest pewny, więc wartość `u` niczego nie zmienia.
Porównaj tabelę i sumę nagród z tym, co masz na kartce.

Współczynnik dyskontowania γ wynosi 0,80. Pierwsza nagroda ma wagę 1,
druga γ, a trzecia γ².

In [ ]:
schedule = [(SEARCH, 0.73), (SEARCH, 0.90), (WAIT, 0.10)]
state = H
rewards = []

for step, (action, u) in enumerate(schedule, start=1):
    next_state, reward = step_robot(model[state, action], u)
    print(f"{step}. {STATE_NAMES[state]} + {action}, u={u:.2f} → "
          f"{STATE_NAMES[next_state]}, nagroda {reward:+g}")
    rewards.append(reward)
    state = next_state

gamma = 0.80
partial_return = sum(gamma**i * reward for i, reward in enumerate(rewards))
print(f"Zdyskontowana suma trzech nagród: {partial_return:.2f}")

## 5. Sprawdź ręczny rachunek Bellmana

**Opcjonalnie po rozwiązaniu układu równań.** Wpisz własne `V_H` i `V_L`.
To oczekiwane zdyskontowane sumy nagród od startu w H lub L przy tej samej
polityce (regule wyboru akcji). W H robot szuka, w L ładuje, a współczynnik
dyskontowania γ wynosi 0,80. Komórka przelicza z tabeli prawą stronę każdego
równania Bellmana, żeby porównać ją z wpisaną wartością.

In [ ]:
from math import isclose

V_H = None  # Wpisz wynik ręcznego rachunku dla H.
V_L = None  # Wpisz wynik ręcznego rachunku dla L.
gamma = 0.80

if V_H is None or V_L is None:
    print("Wpisz obie wartości po rozwiązaniu układu równań.")
else:
    values = {H: V_H, L: V_L}
    policy = {H: SEARCH, L: RECHARGE}

    for evaluated_state, policy_action in policy.items():
        rhs = sum(p * (r + gamma * values[new_state])
                  for p, new_state, r in model[evaluated_state, policy_action])
        print(f"{STATE_NAMES[evaluated_state]}: wpisano {values[evaluated_state]:.4f}; prawa strona {rhs:.4f}")
        assert isclose(values[evaluated_state], rhs, abs_tol=0.01), f"Sprawdź równanie dla {STATE_NAMES[evaluated_state]}."

    print("Oba równania zgadzają się w granicy zaokrąglenia.")